# 06 · Experiment D: sensor + GC-MS fusion in SmellNet, without label leakage (RQ6)

SmellNet's GC-MS data is **one vector per ingredient**. If a test sample's GC-MS vector is looked up by its label, the model is handed the answer. The leakage demonstration below shows this directly. The valid systems use GC-MS only during training (D1, D2) or as the fixed prototypes of *all* 50 classes (D1 prediction), and D3 combines decisions.

**Protocol.** Train on days 1–4, select learning rate / λ / α / checkpoint on day 5, test once on day 6. Code: `src/experiments/expD_smellnet.py` (uses SmellNet's own model classes and contrastive loss).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
R = ROOT / 'results'; F = R / 'figures'; A = R / 'ablations'
def show(p, w=650):
    p = Path(p)
    display(Image(filename=str(p), width=w)) if p.exists() else print('missing figure:', p.name)
def table(p, **kw):
    p = Path(p)
    return pd.read_csv(p, **kw) if p.exists() else print('missing table:', p.name)

## Leakage demonstration (invalid by design; never reported as a result)

In [2]:
table(R / 'expD_leakage_demo.csv')

,system,test_acc1,w,p
0,DX-gcms_only (LEAKY),1.000,100,25
1,DX-early_concat (LEAKY),1.000,100,25
2,"DX-early_concat, label-free GC-MS at test",0.052,100,25
3,sensor-only same features (valid),0.448,100,25


## Results vs the SmellNet paper (Table 2, ICLR 2026)
Our accuracies are fractions; `paper_acc1` is in %. D0 ↔ paper's *sensor-only*, D1 ↔ paper's *cross-modal*.

In [3]:
d = table(R / 'expD_sensor_gcms_fusion.csv')
d if d is None else d[['model','w','p','system','n_seeds','test_acc1_mean','test_acc1_std','paper_acc1']]

,model,w,p,system,n_seeds,test_acc1_mean,test_acc1_std,paper_acc1
0,mlp,50,0,D0,3,0.204,0.011,21.9
1,mlp,50,0,D1,3,0.218,0.010,24.1
2,mlp,50,0,D2,3,0.202,0.008,NaN
3,mlp,50,0,D3,3,0.199,0.004,NaN
4,mlp,50,25,D0,3,0.246,0.014,18.2
5,mlp,50,25,D1,3,0.261,0.008,24.7
6,mlp,50,25,D2,3,0.262,0.012,NaN
7,mlp,50,25,D3,3,0.255,0.003,NaN
8,mlp,100,0,D0,3,0.211,0.021,21.0
9,mlp,100,0,D1,3,0.230,0.009,24.3


In [4]:
table(A / 'AB11_with_without_gcms.csv')

,model,w,p,D0,D1,D2,D3,D1-D0,D2-D0,D3-D0
0,mlp,50,0,0.204,0.218,0.202,0.199,0.014,-0.002,-0.005
1,mlp,50,25,0.246,0.261,0.262,0.255,0.016,0.017,0.010
2,mlp,100,0,0.211,0.230,0.208,0.233,0.019,-0.003,0.022
3,mlp,100,25,0.259,0.268,0.259,0.271,0.009,0.000,0.012


In [5]:
table(A / 'AB15_channel_masking.csv')

,model,w,p,system,baseline_acc1,mask_NO2_delta,mask_C2H5OH_delta,mask_VOC_delta,mask_CO_delta,mask_Alcohol_delta,mask_LPG_delta
0,mlp,50,0,D0,0.204,-0.112,-0.102,-0.072,-0.120,-0.026,-0.041
1,mlp,50,0,D1,0.218,-0.133,-0.098,-0.075,-0.146,-0.033,-0.021
2,mlp,50,25,D0,0.246,-0.112,-0.112,-0.157,-0.129,-0.054,-0.042
3,mlp,50,25,D1,0.261,-0.116,-0.123,-0.151,-0.115,-0.057,-0.039
4,mlp,100,0,D0,0.211,-0.112,-0.078,-0.096,-0.127,-0.031,-0.021
5,mlp,100,0,D1,0.230,-0.130,-0.083,-0.111,-0.146,-0.041,-0.032
6,mlp,100,25,D0,0.259,-0.114,-0.122,-0.176,-0.125,-0.046,-0.082
7,mlp,100,25,D1,0.268,-0.122,-0.122,-0.169,-0.112,-0.040,-0.076


In [6]:
table(A / 'AB12_gcms_encoding.csv')

,model,w,p,gcms,test_acc1_mean,test_acc1_std,n
0,mlp,50,0,atom,0.203,0.022,3
1,mlp,50,0,spec,0.218,0.010,3
2,mlp,50,25,atom,0.259,0.007,3
3,mlp,50,25,spec,0.261,0.008,3
4,mlp,100,0,atom,0.211,NaN,1
5,mlp,100,0,spec,0.230,0.009,3
